Run with shape: (1027368, 18)  
gva1 L1.gva1 total_assets employees tfp_wav1 | gmm(gva1, 2:4) gmm(total_assets, 2:3) iv(tfp_wav1) | timedumm  
- With 'collapse': takes just over 1 minute to run

In [1]:
# Install dependencies if needed:
# !pip install "numpy<2.0.0" "pandas<2.2.0" pydynpd

import pandas as pd
import numpy as np
from pydynpd import regression

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

if np.__version__ >= "2.0.0":
    raise ImportError("NumPy version must be less than 2.0.0")
if pd.__version__ >= "2.2.0":
    raise ImportError("Pandas version must be less than 2.2.0")
print("Libraries loaded successfully.")

NumPy version: 1.26.4
Pandas version: 2.1.4
Libraries loaded successfully.


In [3]:
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")

table_panel_name = "working_yearly_with_tfp_wave"
parquet_path = dirs.tmp_dir / f"{table_panel_name}.parquet"
print(f"Parquet file path: {parquet_path}")
df_panel = pd.read_parquet(parquet_path)
if df_panel is None:
    print(f"Failed to load DataFrame from {parquet_path}")
    df_panel = pd.read_csv(dirs.tmp_dir / f"{table_panel_name}.csv")
if df_panel is None:
    raise FileNotFoundError(f"Could not load DataFrame from {parquet_path} or CSV.")
log_vars = ["gva1", "total_assets", "employees"]
df_filtered = (
    df_panel
    .dropna(subset=["gva1", "total_assets", "employees", "tfp_wav1"])
    .drop_duplicates(subset=["registered_number", "year"], keep="last")
    .assign(**{
        f'ln_{p}': df_panel[p].apply(lambda x: np.log(x) if x > 0 else None) for p in log_vars
    })
)
print(f"DataFrame loaded successfully with shape: {df_filtered.shape}")

Parquet file path: C:\Users\lazyst\Files\ucl\Dissertation\model\tmp\working_yearly_with_tfp_wave.parquet
DataFrame loaded successfully with shape: (1027368, 21)


In [4]:
run_lags = [(2, 2), (2, 3), (2, 4), (3, 2), (3, 3), (4, 2)]
iv_depth = 2

for x_lag, ar_lag in run_lags:
    # 1. Define your variables programmatically
    y_var = "ln_gva1"
    lag_dep_var = " ".join([f"L{i}.{y_var}" for i in range(1, ar_lag + 1)])
    standard_factors = ["ln_total_assets", "ln_employees"]
    instr_standard = standard_factors[0]
    x_var = " ".join(["tfp_wav1"] + [f"L{i}.tfp_wav1" for i in range(1, x_lag + 1)])

    # 2. Build the structural equation 
    # Joins the list of standard factors with spaces, and adds the other variables
    structural_eq = f"{y_var} {lag_dep_var} {' '.join(standard_factors)} {x_var}"

    # 3. Build the Instrument Matrix
    # GMM instruments for endogenous/predetermined vars
    gmm_inst = f"gmm({y_var}, {ar_lag + 1}:{ar_lag + iv_depth + 1}) gmm({instr_standard}, {ar_lag + 1}:{ar_lag + iv_depth})"
    # Standard IVs for strictly exogenous vars
    iv_inst = f"iv({x_var})"

    # 4. Build the Options
    options_arr = ["timedumm", "collapse"]
    options = " ".join(options_arr)

    # 5. Concatenate everything using the pydynpd pipe '|' syntax
    command_str = f"{structural_eq} | {gmm_inst} {iv_inst} | {options}"

    # Let's print it to verify it looks exactly right before running
    print("Generated Command String:")
    print(command_str)
    print("-" * 50)

    # Execute estimation
    sys_gmm_model = regression.abond(command_str, df_filtered, ['registered_number', 'year'])

Generated Command String:
ln_gva1 L1.ln_gva1 L2.ln_gva1 ln_total_assets ln_employees tfp_wav1 L1.tfp_wav1 L2.tfp_wav1 | gmm(ln_gva1, 3:5) gmm(ln_total_assets, 3:4) iv(tfp_wav1 L1.tfp_wav1 L2.tfp_wav1) | timedumm collapse
--------------------------------------------------
 Dynamic panel-data estimation, two-step system GMM
 Group variable: registered_number                       Number of obs = 603058  
 Time variable: year                                     Min obs per group: 0    
 Number of instruments = 28                              Max obs per group: 16   
 Number of groups = 139162                               Avg obs per group: 4.33 
+-----------------+------------+---------------------+-------------+-----------+-----+
|     ln_gva1     |   coef.    | Corrected Std. Err. |      z      |   P>|z|   |     |
+-----------------+------------+---------------------+-------------+-----------+-----+
|    L1.ln_gva1   | -0.0111766 |      0.1167748      |  -0.0957111 | 0.9237500 |     |
